# ?????? ????????????? ???????? ?? sklift.models ? sklearn.pipeline

<br>
<center>
    <a href="https://colab.research.google.com/github/maks-sh/scikit-uplift/blob/master/notebooks/pipeline_usage_RU.ipynb">
        <img src="https://colab.research.google.com/assets/colab-badge.svg">
    </a>
    <br>
    <b><a href="https://github.com/maks-sh/scikit-uplift/">SCIKIT-UPLIFT REPO</a> | </b>
    <b><a href="https://scikit-uplift.readthedocs.io/en/latest/">SCIKIT-UPLIFT DOCS</a> | </b>
    <b><a href="https://scikit-uplift.readthedocs.io/en/latest/user_guide/index.html">USER GUIDE</a></b>
    <br>
    <b><a href="https://nbviewer.jupyter.org/github/maks-sh/scikit-uplift/blob/master/notebooks/pipeline_usage_EN.ipynb">ENGLISH VERSION</a></b>

</center>

? ?????? ???????? ?????????? ??????? ?????? ?????????? ?????? ?? ???????? ??????????????? uplift  ? [sklearn.pipeline](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.pipeline).

?????? ??? ??????? ????? ?? [MineThatData E-Mail Analytics And Data Mining Challenge dataset by Kevin Hillstrom](https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html). ???? ????? ?????? ???????? 64 000 ????????, ??????? ? ????????? ??? ????????? ??????? ? ??????? ?????????? ???????. ????? ???????? ???? ????????? ????????? ???????? ? ??????? email ????????:

* 1/3 ???????? ???? ??????? ????????? ??????? ??? ????????? ???????????? ??????, ?????????????? ??????? ?????????;
* 1/3 ???????? ???? ??????? ????????? ??????? ??? ????????? ???????????? ??????, ?????????????? ??????? ?????????;
* ? ?????????? 1/3 ???????????? ?? ?????????.

??? ??????? ??????? ?? ??????? ???????? ???? ???????? ?? ?????? ? ??????, ???? ?????????? ??????? ? ????? ???? ?? ??? ??????, ????????? ????? ????????? ??????.

?????? ???????? ???????? ?????? ????? ?? [??????](https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html).

????????? ??????????? ??????????:

In [ ]:
# pip install scikit-uplift xgboost==1.0.2 category_encoders==2.1.0 -U

??? ???????? ??????? ??????? ?????? ??? ???????? ?????????????:
* ???, ???? ??????????? ?? ??????????? ????? ????????? ???????? ? ???????? ??????? ???????;
* ???, ???? ?? ??????????? ????????? ????????.

? ???????? ??????? ?????????? ????? ???????????? ?????????? `visit`.

In [ ]:
import pandas as pd
from sklift.datasets import fetch_hillstrom


%matplotlib inline

bunch = fetch_hillstrom(target_col='visit')

dataset, target, treatment = bunch['data'], bunch['target'], bunch['treatment']

print(f'?????? ???????? ?? ?????????: {dataset.shape}')

# Selecting two segments
dataset = dataset[treatment!='Mens E-Mail']
target = target[treatment!='Mens E-Mail']
treatment = treatment[treatment!='Mens E-Mail'].map({
    'Womens E-Mail': 1,
    'No E-Mail': 0
})

print(f'?????? ???????? ????? ?????????: {dataset.shape}')
dataset.head()

???????? ??? ?????? ?? ????????? ? ????????????? ???????:

In [ ]:
from sklearn.model_selection import train_test_split


X_tr, X_val, y_tr, y_val, treat_tr, treat_val = train_test_split(
    dataset, target, treatment, test_size=0.5, random_state=42
)

Select categorical features:

In [ ]:
cat_cols = X_tr.select_dtypes(include='object').columns.tolist()
print(cat_cols)

???????? ?????? ??????? ? ????????? ?? ? pipieline.

In [ ]:
from sklearn.pipeline import Pipeline
from category_encoders import CatBoostEncoder
from sklift.models import ClassTransformation
from xgboost import XGBClassifier


encoder = CatBoostEncoder(cols=cat_cols)
estimator = XGBClassifier(max_depth=2, random_state=42)
ct = ClassTransformation(estimator=estimator)

my_pipeline = Pipeline([
    ('encoder', encoder),
    ('model', ct)
])

<b>??????? pipeline ????? ??? ??????, ?? ??????? treatment ??????? ??? ???????? ???? model: `model__treatment`.</b>

In [ ]:
my_pipeline = my_pipeline.fit(
    X=X_tr,
    y=y_tr,
    model__treatment=treat_tr
)

?????????? uplift ? ????????? uplift@30%

In [ ]:
from sklift.metrics import uplift_at_k


uplift_predictions = my_pipeline.predict(X_val)

uplift_30 = uplift_at_k(y_val, uplift_predictions, treat_val, strategy='overall')
print(f'uplift@30%: {uplift_30:.4f}')